# 01 - Dataset Exploration

**Purpose:** load, validate, and present the dataset so it can be used directly
in the thesis.

The dataset has **161 mixtures**, each photographed **5 times** (805 images),
and each mixture is a combination of **1-3** of the materials
`CL, GP1, GP2, MH1, MH2, SP`. Labels are mixture-level and live in
`data/SoilClassificationTargets.csv`.


In [ ]:
# Make `src` importable and prepare the outputs/ folders.
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, data, targets, splits, metrics, plots

config.ensure_output_dirs()
print("Project root:", ROOT)


## 1. Load the metadata

Semicolon-separated CSV, read with every column as a string first.


In [ ]:
metadata = data.load_metadata()
print("metadata shape:", metadata.shape)
metadata.head()


## 2. Validate the CSV against the image folders

Checks Code uniqueness, missing/extra folders, and that every Code has exactly 5 images.


In [ ]:
report = data.validate_dataset(metadata)
for key, value in report.items():
    print(f"{key}: {value}")


## 3. Build the one-row-per-image dataframe

Mixture-level labels are broadcast to all 5 images of each Code.


In [ ]:
image_df = data.build_image_dataframe(metadata)
print("image_df shape:", image_df.shape)
image_df.head()


## 4. Build and validate the target vectors

`presence` (multi-hot), `mass_fraction`, and `volume_fraction`, all over the global material vocabulary.


In [ ]:
vocabulary = targets.build_material_vocabulary(metadata)
print("Material vocabulary:", vocabulary)

target_vectors = targets.build_targets(metadata, vocabulary=vocabulary)
targets.validate_targets(target_vectors)
print("Targets built and validated.")
print("presence shape       :", target_vectors["presence"].shape)
print("mass_fraction shape  :", target_vectors["mass_fraction"].shape)
print("volume_fraction shape:", target_vectors["volume_fraction"].shape)


## 5. Dataset summary


In [ ]:
presence = target_vectors["presence"]
mixture_sizes = presence.sum(axis=1).astype(int)

print("Number of mixtures :", len(target_vectors["codes"]))
print("Number of images   :", len(image_df))
print("Number of materials:", len(vocabulary))
print("Material vocabulary :", vocabulary)
print()

counts = image_df.groupby("Code").size()
print("Images per Code: min={}, max={}, mean={:.2f}".format(
    counts.min(), counts.max(), counts.mean()))
print()

sizes, freqs = np.unique(mixture_sizes, return_counts=True)
for s, f in zip(sizes, freqs):
    print(f"Mixtures with {s} material(s): {f}")


## 6. Visualize the dataset

Figures are saved under `outputs/figures/`.


In [ ]:
plots.plot_material_distribution(target_vectors, save_path="01_material_distribution.png")
plt.show()

plots.plot_mixture_size_distribution(target_vectors, save_path="01_mixture_size_distribution.png")
plt.show()


In [ ]:
plots.plot_example_images(image_df, n_codes=8, seed=config.RANDOM_SEED,
                          save_path="01_example_images.png")
plt.show()


## 7. Inspect one example mixture

Raw CSV row alongside the presence, mass-fraction, and volume-fraction vectors.


In [ ]:
example_code = "82"  # a 3-material mixture: SP, MH2, MH1
i = target_vectors["codes"].index(example_code)

print("Raw CSV row for Code", example_code)
print(metadata[metadata["Code"] == example_code].T)
print()

inspection = pd.DataFrame({
    "material": vocabulary,
    "presence": target_vectors["presence"][i],
    "mass_fraction": target_vectors["mass_fraction"][i].round(4),
    "volume_fraction": target_vectors["volume_fraction"][i].round(4),
    "density": target_vectors["density_matrix"][i],
})
inspection


## 8. Grouped split sanity check

The core methodological rule: **splits are grouped by `Code`** so that all 5
images of a mixture stay together. The assertions below fail loudly if any
mixture were split across train/val/test.


In [ ]:
train_df, val_df, test_df = splits.grouped_train_val_test_split(
    image_df,
    group_col="Code",
    test_size=config.TEST_SIZE,
    val_size=config.VAL_SIZE,
    seed=config.RANDOM_SEED,
)

print("Rows  -> train:", len(train_df), "val:", len(val_df), "test:", len(test_df))
print("Codes -> train:", train_df["Code"].nunique(),
      "val:", val_df["Code"].nunique(),
      "test:", test_df["Code"].nunique())

splits.check_no_leakage(train_df, val_df, test_df)
splits.check_groups_intact([train_df, val_df, test_df], image_df)
print("OK: no leakage - every Code (all 5 images) stays in a single split.")


## Summary

The dataset loads cleanly: 161 mixtures, 805 images, 6 materials, with valid
presence / mass-fraction / volume-fraction targets and a leakage-free grouped
split. This notebook is the starting point for the three modelling notebooks
(02-05).
